# MusicBrainz obogaćivanje Spotify pesama

Ovaj notebook za svaku pesmu iz `spotify_clean.csv` traži odgovarajući snimak na MusicBrainz-u i izvlači:

`spotify_track_id, mbid, isrc, release_date, release_country, label, artist_type, gender, artist_country, artist_lifespan_begin, artist_lifespan_end, tags`

Rezultat se čuva u `musicbrainz.csv`.

**Logika:**
- Pretraga snimka po `track_name` + `artist_name` (MusicBrainz search API).
- Biranje najboljeg kandidata: prioritet poklapanje trajanja (`duration_ms`) u granicama tolerancije, zatim MB relevance score. Kandidati ispod `MIN_SCORE` se odbacuju.
- `release_date` / `release_country` se uzimaju sa **najstarijeg (originalnog)** release-a; `label` se dobija posebnim pozivom na taj release (MusicBrainz ne dozvoljava `labels` include direktno na recording-u).
- `tags` = tagovi na nivou **pesme** (recording tags) - MB snimci retko imaju tagove, pa je često prazno polje, to je očekivano.
- Podaci o izvođaču se keširaju po MBID-u.
- Poštuje se MusicBrainz rate limit (~1 zahtev/sekundi)

## 1. Instalacija biblioteke

In [ ]:
!pip install -q musicbrainzngs

## 2. Podešavanja i uvoz

In [ ]:
import csv
import sys
import time

import musicbrainzngs as mb

APP_NAME = "SpotifyMusicBrainzEnrichment"
APP_VERSION = "1.0"
CONTACT = "primer@example.com"  # MusicBrainz traži kontakt u User-Agent-u

INPUT_CSV = "spotify_clean.csv"
MIN_SCORE = 60                 # minimalni MB search score (0-100)
DURATION_TOLERANCE_MS = 10000  # tolerancija poklapanja trajanja
SLEEP_SECONDS = 1.1            # pauza između zahteva (MB rate limit ~1/sec)

FIELDNAMES = [
    "spotify_track_id", "mbid", "isrc", "release_date", "release_country",
    "label", "artist_type", "gender", "artist_country",
    "artist_lifespan_begin", "artist_lifespan_end", "tags",
]

mb.set_useragent(APP_NAME, APP_VERSION, CONTACT)
mb.set_rate_limit(limit_or_interval=1.0, new_requests=1)
print("MusicBrainz klijent podešen.")

## 3. Pomoćne funkcije (matching logika)

In [ ]:
def duration_matches(mb_length_ms, spotify_ms, tolerance_ms):
    if mb_length_ms is None or spotify_ms is None or spotify_ms == "":
        return False
    try:
        return abs(int(mb_length_ms) - int(float(spotify_ms))) <= tolerance_ms
    except (TypeError, ValueError):
        return False


def pick_best_recording(candidates, spotify_ms, tolerance_ms, min_score):
    """Vrati najbolji kandidat: prioritet poklapanje trajanja, zatim score."""
    scored = []
    for rec in candidates:
        try:
            score = int(rec.get("ext:score", rec.get("score", 0)) or 0)
        except (TypeError, ValueError):
            score = 0
        if score < min_score:
            continue
        match = duration_matches(rec.get("length"), spotify_ms, tolerance_ms)
        scored.append((match, score, rec))
    if not scored:
        return None
    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return scored[0][2]


def earliest_release(release_list):
    """Vrati release sa najranijim poznatim datumom (YYYY ili YYYY-MM-DD)."""
    dated = [r for r in (release_list or []) if r.get("date")]
    if not dated:
        return None

    def sort_key(r):
        parts = r["date"].split("-")
        parts = parts + ["01"] * (3 - len(parts))
        try:
            return tuple(int(p) for p in parts[:3])
        except ValueError:
            return (9999, 12, 31)

    dated.sort(key=sort_key)
    return dated[0]


def get_release_info(release):
    """Vrati (date, country) sa release-a. Label se dobija posebnim pozivom
    jer 'labels' nije validan include na nivou recording-a, samo release-a."""
    if not release:
        return "", ""
    date = release.get("date", "")
    country = release.get("country", "")
    if not country:
        for ev in release.get("release-event-list", []) or []:
            area = ev.get("area", {}) or {}
            codes = area.get("iso-3166-1-code-list")
            if codes:
                country = codes[0]
                break
    return date, country


def fetch_label_for_release(release_id):
    if not release_id:
        return ""
    try:
        rel_full = mb.get_release_by_id(release_id, includes=["labels"])["release"]
    except mb.WebServiceError as e:
        print(f"  [warn] label lookup za release {release_id} nije uspeo: {e}", file=sys.stderr)
        return ""
    labels = []
    for li in rel_full.get("label-info-list", []) or []:
        name = (li.get("label", {}) or {}).get("name")
        if name:
            labels.append(name)
    return ";".join(dict.fromkeys(labels))


def fetch_artist_info(artist_id, cache):
    if artist_id in cache:
        return cache[artist_id]
    info = {
        "artist_type": "", "gender": "", "artist_country": "",
        "artist_lifespan_begin": "", "artist_lifespan_end": "",
    }
    for attempt in range(3):
        try:
            result = mb.get_artist_by_id(artist_id, includes=["tags"])
            artist = result.get("artist", {})
            info["artist_type"] = artist.get("type", "")
            info["gender"] = artist.get("gender", "")
            info["artist_country"] = (
                artist.get("country", "") or (artist.get("area", {}) or {}).get("name", "")
            )
            lifespan = artist.get("life-span", {}) or {}
            info["artist_lifespan_begin"] = lifespan.get("begin", "")
            info["artist_lifespan_end"] = lifespan.get("end", "")
            break
        except mb.WebServiceError as e:
            print(f"  [warn] artist lookup {artist_id} nije uspeo (pokušaj {attempt + 1}): {e}", file=sys.stderr)
            time.sleep(2)
    cache[artist_id] = info
    return info


def process_track(row, duration_tolerance_ms, min_score, artist_cache):
    track_id = row["track_id"]
    track_name = row["track_name"]
    artist_name = row["artist_name"]
    spotify_ms = row.get("duration_ms")

    out = {fn: "" for fn in FIELDNAMES}
    out["spotify_track_id"] = track_id

    try:
        search = mb.search_recordings(recording=track_name, artist=artist_name, limit=10)
    except mb.WebServiceError as e:
        print(f"  [warn] pretraga nije uspela za {track_id} ({track_name} - {artist_name}): {e}", file=sys.stderr)
        return out, False

    candidates = search.get("recording-list", [])
    best = pick_best_recording(candidates, spotify_ms, duration_tolerance_ms, min_score)
    if not best:
        return out, False

    mbid = best.get("id", "")
    out["mbid"] = mbid

    try:
        full = mb.get_recording_by_id(
            mbid,
            includes=["isrcs", "tags", "releases", "artist-credits"],
        )["recording"]
    except mb.WebServiceError as e:
        print(f"  [warn] detalji snimka {mbid} nisu preuzeti: {e}", file=sys.stderr)
        full = best  # radi sa onim što imamo iz pretrage

    isrcs = full.get("isrc-list", []) or []
    out["isrc"] = ";".join(isrcs)

    tags = full.get("tag-list", []) or []
    out["tags"] = ";".join(t.get("name", "") for t in tags if t.get("name"))

    rel = earliest_release(full.get("release-list", []))
    date, country = get_release_info(rel)
    label = fetch_label_for_release(rel.get("id") if rel else None)
    out["release_date"], out["release_country"], out["label"] = date, country, label

    artist_id = None
    for ac in full.get("artist-credit", []) or []:
        if isinstance(ac, dict) and "artist" in ac:
            artist_id = ac["artist"].get("id")
            break
    if artist_id:
        out.update(fetch_artist_info(artist_id, artist_cache))

    return out, True


def run(input_csv, output_csv, limit=None, min_score=MIN_SCORE,
        duration_tolerance_ms=DURATION_TOLERANCE_MS, sleep_seconds=SLEEP_SECONDS):
    with open(input_csv, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if limit:
        rows = rows[:limit]

    artist_cache = {}
    found = 0
    with open(output_csv, "w", newline="", encoding="utf-8") as out_f:
        writer = csv.DictWriter(out_f, fieldnames=FIELDNAMES)
        writer.writeheader()
        for i, row in enumerate(rows, 1):
            print(f"[{i}/{len(rows)}] {row['artist_name']} - {row['track_name']}")
            result, ok = process_track(row, duration_tolerance_ms, min_score, artist_cache)
            writer.writerow(result)
            out_f.flush()
            found += int(ok)
            time.sleep(sleep_seconds)

    print(f"\nGotovo. Pronađeno {found}/{len(rows)} pesama. Rezultat: {output_csv}")
    return output_csv

print("Funkcije definisane.")

## 5. Pokreni na CELOM fajlu

Ovo obrađuje svih 2145 pesama i traje otprilike 1.5-2h (MusicBrainz rate limit ~1 zahtev/sekundi, 3 poziva po pronađenoj pesmi). Rezultat ide u `musicbrainz.csv`.

In [ ]:
run(INPUT_CSV, "musicbrainz.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("musicbrainz.csv")

print("Broj redova i kolona:", df.shape)
print("\nTipovi podataka:")
print(df.dtypes)

null_summary = df.isna().sum().to_frame("null_count")
null_summary["null_pct"] = (null_summary["null_count"] / len(df) * 100).round(1)
print("\nNull vrednosti po koloni:")
print(null_summary.sort_values("null_pct", ascending=False))

print("\nBroj jedinstvenih vrednosti po koloni:")
print(df.nunique())

df.head()

In [ ]:
dupes = df[df.duplicated(subset=["mbid"], keep=False)].sort_values("mbid")
dupes[["spotify_track_id", "mbid"]]

In [ ]:
import pandas as pd

mb_df = pd.read_csv("musicbrainz.csv")
sp = pd.read_csv("spotify_clean.csv")

# redovi čiji mbid nije jedinstven
dupes = mb_df[mb_df.duplicated(subset=["mbid"], keep=False)].sort_values("mbid")

# spoji sa spotify podacima da vidiš track_name/artist_name/album_name
dupes_full = dupes.merge(
    sp[["track_id", "track_name", "artist_name", "album_name", "album_type", "duration_ms"]],
    left_on="spotify_track_id",
    right_on="track_id",
    how="left",
)

cols = ["mbid", "spotify_track_id", "track_name", "artist_name", "album_name", "album_type", "duration_ms"]
print(f"Broj redova sa deljenim mbid: {len(dupes_full)}")
dupes_full[cols].sort_values(["mbid", "duration_ms"])

In [ ]:
import pandas as pd
import time

df = pd.read_csv("musicbrainz.csv")
sp = pd.read_csv("spotify_clean.csv")

suspect_ids = df.loc[df.duplicated(subset=["mbid"], keep=False), "spotify_track_id"].tolist()
retry_rows = sp[sp["track_id"].isin(suspect_ids)]
print(f"Sumnjivih pesama: {len(retry_rows)}")

def show_candidates(track_name, artist_name, spotify_ms, limit=10):
    search = mb.search_recordings(recording=track_name, artist=artist_name, limit=limit)
    rows = []
    for rec in search.get("recording-list", []):
        score = int(rec.get("ext:score", rec.get("score", 0)) or 0)
        length = rec.get("length")
        diff_s = abs(int(length) - int(spotify_ms)) / 1000 if length and spotify_ms else None
        artist_credit = ", ".join(
            ac["artist"]["name"] for ac in rec.get("artist-credit", [])
            if isinstance(ac, dict) and "artist" in ac
        )
        rows.append({
            "mbid": rec.get("id"), "title": rec.get("title"), "artist": artist_credit,
            "score": score,
            "length_s": round(int(length) / 1000, 1) if length else None,
            "diff_s": round(diff_s, 1) if diff_s is not None else None,
        })
    return pd.DataFrame(rows).sort_values("score", ascending=False)

for _, row in retry_rows.iterrows():
    print("=" * 90)
    print(f"{row['track_name']} — {row['artist_name']}  "
          f"(track_id={row['track_id']}, trajanje={row['duration_ms']/1000:.1f}s)")
    print(show_candidates(row["track_name"], row["artist_name"], row["duration_ms"]).to_string(index=False))
    time.sleep(1.1)

In [ ]:
corrected_mbids = {
    # pronađene ispravne pesme (tačan naslov + diff_s ~0)
    "3VfbbBRtjIjrDyElXWJzI8": "21e63ad3-1a8b-4f49-b77a-5733b49ab572",  # O Mi To Fu
    "2aRIH8e5e4Lx0hsCU86J3w": "82e3ac36-b37d-414a-ae22-9e903ea6728e",  # Hollow Ponds
    "0Vu6WUisw1MwcZUVKIrn5l": "98c0f7b3-0d12-47ad-a706-5956fc6454be",  # The Tower Of Montevideo
    "3nfItYZl5QhfquxCe0RKSE": "69dfdd6e-c527-4714-afd6-e8cdabac4bd6",  # The Cormorant
    "4od6yztrJvUwjj5nyECvAV": "9d6f3f5e-c283-4024-9f2f-3b281101163e",  # Giraffe Trumpet Sea
    "1P8ii7oHWJFv6cCEvceBaO": "2689ad41-15d4-420a-82fb-1a1811380b7c",  # Fuckingsong
    "2W7FcHE1RLpbJ5RbEtXYDG": "dd34feaa-fb57-41c8-a139-f4f3b5162e29",  # Sometimes I Am Pharaoh
    "5JsVlXDFbiSzWIZ7cbxwEg": "3bb008f1-6531-459d-9e18-8ee10014808a",  # Fat Children (ima 2 identična MB zapisa, izabran jedan)

    # nema pouzdanog poklapanja u top 10 - MB verovatno nema ove numere katalogizovane
    "6SR69waFd4pU2oSJ2xrYvZ": None,   # Les ecrocs
    "6j1eq9RFweno8xfGRi9GUf": None,   # Monkey's World
    "6hn5tLZaF69knJq3JqWBX0": None,   # Monkey Travels
    "6txZb4vBRgmQdePx9olXX8": None,   # Into The Eastern Sea
    "0JJolcxh3vNm7B3xxscsat": None,   # The Living Sea
    "13nOO4MSxvsq00sB8QZhkB": None,   # The Dragon King
    "6SD09CgdK54xcOuiBuT2kD": None,   # Sandy The River Demon
    "2mTaIdJsL31nlCpAxOPDj9": None,   # The White Skeleton Demon
    "6Y8ZoQqEiUwe3dkAQ5ktE5": None,   # March Of The Iron Army
    "4hszF784jEq3JTz2ktVpJS": None,   # Monkey Bee
    "0SrPq63zeMkZmqXsMNAr8V": None,   # Am I Missing Something?
    "0KacdizqDsSrzWBg70RnQg": None,   # Children of the Echo
    "2dWrIJdwYWI6K7WWL3kpqu": None,   # Contact
}

In [ ]:
if "artist_cache" not in globals():
    artist_cache = {}

mb_columns = [
    "mbid", "isrc", "release_date", "release_country", "label",
    "artist_type", "gender", "artist_country",
    "artist_lifespan_begin", "artist_lifespan_end", "tags",
]

for track_id, new_mbid in corrected_mbids.items():
    idx = df.index[df["spotify_track_id"] == track_id][0]

    if new_mbid is None:
        df.loc[idx, mb_columns] = pd.NA
        continue

    full = mb.get_recording_by_id(
        new_mbid, includes=["isrcs", "tags", "releases", "artist-credits"]
    )["recording"]

    df.loc[idx, "mbid"] = new_mbid
    df.loc[idx, "isrc"] = ";".join(full.get("isrc-list", []) or [])
    df.loc[idx, "tags"] = ";".join(t.get("name", "") for t in full.get("tag-list", []) or [] if t.get("name"))

    rel = earliest_release(full.get("release-list", []))
    date, country = get_release_info(rel)
    df.loc[idx, "release_date"] = date
    df.loc[idx, "release_country"] = country
    df.loc[idx, "label"] = fetch_label_for_release(rel.get("id") if rel else None)

    artist_id = next(
        (ac["artist"]["id"] for ac in full.get("artist-credit", []) if isinstance(ac, dict) and "artist" in ac),
        None,
    )
    if artist_id:
        info = fetch_artist_info(artist_id, artist_cache)
        for k, v in info.items():
            df.loc[idx, k] = v

    time.sleep(1.1)

df.to_csv("musicbrainz.csv", index=False)
print("musicbrainz.csv ažuriran.")
df["mbid"].nunique(), df["mbid"].isna().sum()

# Čišćenje musicbrainz.csv

U ovom delu se čiste podaci dobijeni sa MusicBrainz-a pre spajanja sa Spotify datasetom (spajanje NIJE deo ovog notebook-a).

**Odluke o čišćenju:**
- `isrc` kolona se **potpuno izbacuje** — spotify_track_id već postoji kao ključ za spajanje, a isrc ima ~38.7% null vrednosti i ionako se ne koristi za merge.
- `label`, `artist_type`, `artist_country` → prazno postaje `"Unknown"`.
- `gender` → `"N/A (Group)"` kod izvođača tipa `Group` (MusicBrainz ne beleži pol za bendove — nije primenjivo), `"Unknown"` samo kod izvođača kod kojih je pol stvarno nepoznat.
- `artist_lifespan_begin`/`artist_lifespan_end` ostaju kao originalne vrednosti; dodata je kolona `is_active` (da li je izvođač i dalje aktivan/živ) izračunata direktno iz njih.
- `label` i `tags` su multi-value kolone spojene sa `;` — dodate su pomoćne kolone `has_label`, `num_tags`, `has_tags`.
- `tags` i `release_country` — null vrednosti se zamenjuju sa `"Unknown"` (release_country se inače ni na koji drugi način ne dira — nema mapiranja MB pseudo-kodova poput XW/XE, samo se prazna polja pune).
- sve kolone dobijaju prefiks `musicbrainz_`, **osim** `spotify_track_id` — ovo je poslednji korak, neposredno pre snimanja, da ne bi pokvario prethodni kod koji koristi originalna imena kolona.

## 1. Učitavanje podataka (i izbacivanje isrc kolone)

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("musicbrainz.csv")
df = df.drop(columns=["isrc"])
print(f"Broj redova: {len(df)}")
df.head()

Broj redova: 2145


,spotify_track_id,mbid,release_date,release_country,label,artist_type,gender,artist_country,artist_lifespan_begin,artist_lifespan_end,tags
0,4Kucn8p48pDkoZdIRCOB01,098f842d-6628-4978-94f3-e152b1f1aed0,NaN,NaN,NaN,Group,NaN,GB,1989,NaN,NaN
1,0boNKrTLvgY200vkhmfkrl,96388e46-7362-40bd-979c-d16e08172e72,1994-10-01,XW,Echo,Group,NaN,GB,1989,NaN,NaN
2,3dvxpgCNCgZqSqAdnSNU7n,ee26fb03-16fe-4737-9e83-7189da3a16d0,1994-10-01,XW,Echo,Group,NaN,GB,1989,NaN,alternative rock;indie;indie rock;pop rock;pun...
3,1o5gIYys3cw11Ryur8GrND,accceed3-5be4-4520-a3b4-56885c7404e6,2019-03-22,GB,Edsel Records,Group,NaN,GB,1989,NaN,NaN
4,37LhwN5oJmvVmhBpl4MpSA,2fffca52-32c9-4e17-a4d1-d08a92f290b0,1994-10-01,XW,Echo,Group,NaN,GB,1989,NaN,britpop;indie;indie rock;punk;rock


## 2. Null vrednosti pre čišćenja

In [2]:
def null_summary(frame):
    s = frame.isna().sum().to_frame("null_count")
    s["null_pct"] = (s["null_count"] / len(frame) * 100).round(1)
    return s.sort_values("null_pct", ascending=False)

null_summary(df)

,null_count,null_pct
artist_lifespan_end,1819,84.8
gender,1637,76.3
tags,1045,48.7
release_country,256,11.9
label,199,9.3
release_date,81,3.8
artist_country,29,1.4
artist_lifespan_begin,28,1.3
artist_type,14,0.7
mbid,13,0.6


## 3. Pomoćne kolone za multi-value polja (label, tags)

Rade se **pre** popunjavanja Unknown vrednosti, na osnovu originalnih (sirovih) podataka.

In [3]:
def count_values(value, sep=";"):
    if pd.isna(value) or str(value).strip() == "":
        return 0
    return len(str(value).split(sep))

df["has_label"] = df["label"].apply(count_values) > 0

df["num_tags"] = df["tags"].apply(count_values)
df["has_tags"] = df["num_tags"] > 0

df[["has_label", "num_tags", "has_tags"]].describe(include="all")

,has_label,num_tags,has_tags
count,2145,2145.000000,2145
unique,2,NaN,2
top,True,NaN,True
freq,1946,NaN,1100
mean,NaN,2.642890,NaN
std,NaN,3.598459,NaN
min,NaN,0.000000,NaN
25%,NaN,0.000000,NaN
50%,NaN,1.000000,NaN
75%,NaN,5.000000,NaN


## 4. Kategoričke kolone → "Unknown" (label, artist_country, artist_type)

`release_country` se ovde **ne dira** — puni se posebno u koraku 7, samo null vrednosti.

In [4]:
df["label"] = df["label"].fillna("Unknown")
df["artist_country"] = df["artist_country"].fillna("Unknown")
df["artist_type"] = df["artist_type"].fillna("Unknown")

df[["label", "artist_country", "artist_type"]].head()

,label,artist_country,artist_type
0,Unknown,GB,Group
1,Echo,GB,Group
2,Echo,GB,Group
3,Edsel Records,GB,Group
4,Echo,GB,Group


## 5. Gender — razlikovanje "nije primenjivo" (Group) od stvarno nepoznatog

In [5]:
def resolve_gender(row):
    if pd.notna(row["gender"]):
        return row["gender"]
    if row["artist_type"] == "Group":
        return "N/A (Group)"
    return "Unknown"

df["gender"] = df.apply(resolve_gender, axis=1)
df["gender"].value_counts(dropna=False)

gender
N/A (Group)       1621
Male               506
Unknown             16
Female               1
Not applicable       1
Name: count, dtype: int64

## 6. is_active — da li je izvođač i dalje aktivan/živ

Računa se direktno iz `artist_lifespan_begin`/`artist_lifespan_end`. `NaN` ako ne znamo ni kad je karijera počela, `True` ako nema kraja (aktivan/živ), `False` ako kraj postoji.

In [6]:
def compute_is_active(row):
    if pd.isna(row["artist_lifespan_begin"]):
        return np.nan
    return pd.isna(row["artist_lifespan_end"])

df["is_active"] = df.apply(compute_is_active, axis=1)
df["is_active"].value_counts(dropna=False)

is_active
True     1791
False     326
NaN        28
Name: count, dtype: int64

## 7. Popunjavanje null vrednosti u tags i release_country sa "Unknown"

In [7]:
df["tags"] = df["tags"].fillna("Unknown")
df["release_country"] = df["release_country"].fillna("Unknown")

df[["tags", "release_country"]].isna().sum()

tags               0
release_country    0
dtype: int64

## 8. Null vrednosti posle čišćenja

In [8]:
null_summary(df)

,null_count,null_pct
artist_lifespan_end,1819,84.8
release_date,81,3.8
artist_lifespan_begin,28,1.3
is_active,28,1.3
mbid,13,0.6
release_country,0,0.0
spotify_track_id,0,0.0
gender,0,0.0
artist_type,0,0.0
label,0,0.0


## 9. Preimenovanje kolona — prefiks musicbrainz_ (osim spotify_track_id)

Poslednji korak pre snimanja, da ne pokvari prethodni kod koji koristi originalna imena kolona.

In [9]:
df = df.rename(columns={col: f"musicbrainz_{col}" for col in df.columns if col != "spotify_track_id"})
df.columns.tolist()

['spotify_track_id',
 'musicbrainz_mbid',
 'musicbrainz_release_date',
 'musicbrainz_release_country',
 'musicbrainz_label',
 'musicbrainz_artist_type',
 'musicbrainz_gender',
 'musicbrainz_artist_country',
 'musicbrainz_artist_lifespan_begin',
 'musicbrainz_artist_lifespan_end',
 'musicbrainz_tags',
 'musicbrainz_has_label',
 'musicbrainz_num_tags',
 'musicbrainz_has_tags',
 'musicbrainz_is_active']

## 10. Snimanje očišćenog fajla

In [10]:
df.to_csv("musicbrainz_clean.csv", index=False)
print("Sačuvano: musicbrainz_clean.csv")
df.shape

Sačuvano: musicbrainz_clean.csv


(2145, 15)